In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import networkx as nx
import matplotlib.pyplot as plt
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw

from tokengt_paper_repo.tokengt_paper import TokenGTPaperGraphRegression
from tokengt_paper_experiments.pcqm4m_dataset import PCQM4MDataset
from models.add_smarts_instances import get_qm9_smarts_patterns

In [2]:
def load_model_and_checkpoint(checkpoint_path, config):
    """Load the model from checkpoint"""
    smarts_patterns = get_qm9_smarts_patterns()
    n_substructures = len(smarts_patterns)

    data_module = PCQM4MDataset(
        batch_size=1,
        num_workers=config["num_workers"],
        d_p=config["D_P"],
        node_id_mode=config["node_id_mode"],
        smarts_patterns=smarts_patterns,
        embed_smarts=config["embed_smarts"],
        dataset_fraction=0.01,
    )

    data_module.setup()
    
    node_dim = 9 + n_substructures if config["embed_smarts"] else 9
    num_atoms = PCQM4MDataset.SINGLE_EMB_OFFSET * node_dim
    num_edges = 4 * PCQM4MDataset.SINGLE_EMB_OFFSET
    
    if config["architecture"] == "TokenGT_Paper":
        model = TokenGTPaperGraphRegression(
            num_atoms=num_atoms,
            num_edges=num_edges,
            d_p=config["D_P"],
            d=config["d"],
            num_heads=config["num_heads"],
            num_encoder_layers=config["num_encoder_layers"],
            node_id_mode=config["node_id_mode"],
            dropout=config["dropout"],
            lr=config["lr"],
            batch_size=config["batch_size"],
            weight_decay=config["weight_decay"],
            return_attention=True,
            use_interaction_bias=config["use_interaction_bias"],
        )
    elif config["architecture"] == "TokenGT_Paper_Sum":
        model = TokenGTPaperGraphRegression(
            num_atoms=num_atoms,
            num_edges=num_edges,
            d_p=config["D_P"],
            d=config["d"],
            num_heads=config["num_heads"],
            num_encoder_layers=config["num_encoder_layers"],
            node_id_mode=config["node_id_mode"],
            dropout=config["dropout"],
            lr=config["lr"],
            batch_size=config["batch_size"],
            weight_decay=config["weight_decay"],
            substructure_mode="sum",
            n_substructures=n_substructures,
            return_attention=True,
            use_interaction_bias=config["use_interaction_bias"],
        )
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    
    return model, data_module

config = {
    "architecture": "TokenGT_Paper_Sum",
    "dataset": "PCQM4M",
    "embed_smarts": False,
    "node_id_mode": "orf",
    "D_P": 16,
    "num_heads": 16,
    "d": 128,
    "num_encoder_layers": 4,
    "epochs": 100,
    "batch_size": 512,
    "lr": 0.005,
    "num_workers": 8,
    "weight_decay": 0.01,
    "dropout": 0.1,
    "checkpointing": False,
    "seed": 1,
    "use_interaction_bias": False
}

checkpoint_path = "tokengt_experiments/model_analysis/tgt_s_orf-val_loss=0.1832.ckpt"
# checkpoint_path = "tokengt_experiments/model_analysis/tgt_s_lap-val_loss=0.1431.ckpt"
# tgts_model, data_module = load_model_and_checkpoint(checkpoint_path, config)

# checkpoint_path = "tokengt_experiments/model_analysis/tgt_orf-val_loss=0.1914.ckpt"
checkpoint_path = "tokengt_experiments/model_analysis/tgt_orf_emb-val_loss=0.1722.ckpt"
config["embed_smarts"] = True
# checkpoint_path = "tokengt_experiments/model_analysis/tgt_lap-val_loss=0.1382.ckpt"
config["architecture"] = "TokenGT_Paper"
tgt_model, data_module = load_model_and_checkpoint(checkpoint_path, config)

Using 1.0% of dataset: train=10, val=1000, test=147037


In [3]:
samples = []
for batch in data_module.val_dataloader():
    samples.append(batch)
    if len(samples) > 100:
        break

print(samples)

[DataBatch(x=[17, 19], edge_index=[2, 36], edge_attr=[36, 3], y=[1], smiles=[1], substructure_instances=[0, 8], n_substructure_instances=[1], node_data=[17, 19], edge_data=[36, 3], in_degree=[17], out_degree=[17], lap_eigvec=[17, 16], batch=[17], ptr=[2]), DataBatch(x=[10, 19], edge_index=[2, 22], edge_attr=[22, 3], y=[1], smiles=[1], substructure_instances=[0, 8], n_substructure_instances=[1], node_data=[10, 19], edge_data=[22, 3], in_degree=[10], out_degree=[10], lap_eigvec=[10, 16], batch=[10], ptr=[2]), DataBatch(x=[17, 19], edge_index=[2, 34], edge_attr=[34, 3], y=[1], smiles=[1], substructure_instances=[7, 8], n_substructure_instances=[1], node_data=[17, 19], edge_data=[34, 3], in_degree=[17], out_degree=[17], lap_eigvec=[17, 16], batch=[17], ptr=[2]), DataBatch(x=[16, 19], edge_index=[2, 34], edge_attr=[34, 3], y=[1], smiles=[1], substructure_instances=[6, 8], n_substructure_instances=[1], node_data=[16, 19], edge_data=[34, 3], in_degree=[16], out_degree=[16], lap_eigvec=[16, 16

In [4]:
tgt_model._token_gt.graph_feature.atom_encoder.weight.shape

torch.Size([9728, 128])

In [5]:
motif_emb = tgt_model._token_gt.graph_feature.atom_encoder.weight[-9*512-1:-1].reshape(9, 512, config["d"])
motif_emb.shape

torch.Size([9, 512, 128])

In [12]:
norms = tgt_model._token_gt.graph_feature.atom_encoder.weight.norm(dim=1)
np.where(norms > 2), norms[np.where(norms > 2)]

((array([   4,    5,    6,    7,    8,   13,   14,   15,   16,   31,   32,
           33,   34,  513,  514, 1024, 1025, 1026, 1027, 1028, 1029, 1030,
         1540, 1542, 2048, 2049, 2050, 2051, 2561, 2562, 2563, 3072, 3073,
         3074, 3075, 3076, 3077, 3585, 4096, 4097, 4609, 5121, 5633, 6145,
         6657, 7169, 7681, 8193, 8705, 9217]),),
 tensor([5.8663, 3.2393, 3.9789, 4.4020, 6.8414, 7.3644, 6.4539, 5.6880, 6.6317,
         6.5487, 6.3699, 8.3075, 7.3101, 5.7518, 5.1051, 5.5997, 4.4303, 3.6131,
         2.5637, 4.2899, 6.2630, 3.4066, 6.4837, 7.0014, 2.8357, 3.3825, 5.1482,
         6.8324, 4.0792, 6.5765, 6.2833, 6.1091, 2.2636, 3.5944, 5.7136, 3.3735,
         3.3812, 3.7837, 2.6300, 2.4993, 6.3073, 4.7192, 5.5718, 4.3128, 4.1395,
         5.2681, 4.4978, 5.5295, 4.7380, 4.6064], grad_fn=<IndexBackward0>))

In [20]:
motif_emb[2, 0].norm()

tensor(0.2086, grad_fn=<LinalgVectorNormBackward0>)

In [14]:
sample_idx = 4
sample = samples[sample_idx]
sample.x, sample.node_data, sample.substructure_instances

(tensor([[6, 0, 1, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [5, 0, 2, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
         [5, 0, 3, 5, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
         [5, 0, 3, 5, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 0, 3, 5, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
         [5, 2, 4, 5, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [6, 0, 2, 5, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
         [5, 0, 3, 5, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
         [7, 0, 2, 5, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
         [5, 0, 4, 5, 3, 0, 2, 0, 0, 0, 0, 0, 0,

In [18]:
sample.x.shape

torch.Size([16, 19])